In [ ]:
import os
import time
import random
import itertools

import numpy as np
import pandas as pd
import math

import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, losses, backend as K
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, Lion

from keras import Sequential
from keras.layers import Dense, Dropout, Input, LeakyReLU
from keras.optimizers import Adam, RMSprop

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    PowerTransformer,
    OneHotEncoder,
    StandardScaler
)
from sklearn.decomposition import PCA
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import (
    IterativeImputer,
    KNNImputer,
    SimpleImputer
)
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

In [ ]:
data= pd.read_excel('Raw Dataset_dir')

In [ ]:
X= data.iloc[:,:31]
y= data.iloc[:,31:32]

In [ ]:
ki= KNNImputer(n_neighbors=2)
X_imputed= pd.DataFrame(ki.fit_transform(X), columns=X.columns)

In [ ]:
sc= StandardScaler()
X_std= pd.DataFrame(sc.fit_transform(X_imputed), columns=X.columns)

In [ ]:
data_std= pd.concat([X_std, pd.DataFrame(y)], axis=1)
plt.rcParams['font.family']= 'Times New Roman'
plt.rcParams['font.size']=18
plt.figure(figsize=(40,30))
sns.heatmap(data_std.corr(), annot=True, linewidths=0.1)
plt.show()

In [ ]:
df_reduced = data_std.drop(['Epoxy Descriptor 4', 'Catalyst Descriptor 2', 
                         'Catalyst Descriptor 4', 'Catalyst (%wt)', 'Alternate Tg (C)', 'Alternate Temperature', 'Temperature (C)',
                         'Required Time (min)', 'Additive Descriptor 5'], axis=1)

In [ ]:
plt.rcParams['font.family']= 'Times New Roman'
plt.rcParams['font.size']=25

plt.figure(figsize=(40,30))
ax= sns.heatmap(df_reduced.corr(), annot=True, linewidths=0.1)

ax.set_xticklabels(ax.get_xticklabels(), fontsize=25, fontname='Times New Roman', rotation=90, ha="left")
ax.set_yticklabels(ax.get_yticklabels(), fontsize=25, fontname='Times New Roman', rotation=0)

plt.show()

In [ ]:
X_std1= df_reduced.drop("Self-healing Effficiency (%)", axis=1)
y1= df_reduced["Self-healing Effficiency (%)"]

In [ ]:
plt.rcParams['font.size']=14

axes = pd.plotting.scatter_matrix(
    X_std1, c=y1, figsize=(25,25), diagonal='hist', 
    cmap="summer", s=100, hist_kwds={'color':(38/255, 102/255, 127/255), 'edgecolor': 'black'}
)

n = len(axes)
cols = X_std1.columns

for i in range(n):
    for j in range(n):
        ax = axes[i, j]

        if i < j:
            x = X_std1.iloc[:, j].to_numpy()
            y = X_std1.iloc[:, i].to_numpy()
            sns.kdeplot(x=x, y=y, ax=ax, levels=10, linewidths=1, color='black')

        if i < n - 1:
            ax.set_xticklabels([])
        if j > 0:
            ax.set_yticklabels([])


for j in range(n):
    axes[n-1, j].set_xlabel(
        cols[j],
        fontsize=20,
        fontname='Times New Roman',
        rotation=45,
        labelpad=0,
        ha="right" 
    )

for i in range(n):
    axes[i, 0].set_ylabel(
        cols[i],
        fontsize=20,
        rotation= 45,
         labelpad=65,
        fontname='Times New Roman'
    )

plt.subplots_adjust(wspace=0, hspace=0)
plt.show()

In [ ]:
pt = PowerTransformer()
X_q = pd.DataFrame(pt.fit_transform(X_std1), columns=X_std1.columns)

In [ ]:
plt.rcParams['font.size']=14

axes = pd.plotting.scatter_matrix(
    X_q, c=y1, figsize=(35,35), diagonal='hist'
)

n = len(axes)
cols = X_q.columns

for i in range(n):
    for j in range(n):
        ax = axes[i, j]

        if i < j:
            x = X_q.iloc[:, j].to_numpy()
            y = X_q.iloc[:, i].to_numpy()
            sns.kdeplot(x=x, y=y, ax=ax, levels=10, linewidths=1, color='black')


        if i < n - 1:
            ax.set_xticklabels([])
        if j > 0:
            ax.set_yticklabels([])

for j in range(n):
    axes[n-1, j].set_xlabel(cols[j])
for i in range(n):
    axes[i, 0].set_ylabel(cols[i])

plt.subplots_adjust(wspace=0, hspace=0)
plt.show()

In [ ]:
pca1 = PCA(n_components=15, random_state=42)
X_dec = pd.DataFrame(pca1.fit_transform(X_q))

In [ ]:
axes = pd.plotting.scatter_matrix(
    X_dec, c=y1, figsize=(25,25), diagonal='hist',hist_kwds={'color':(38/255, 102/255, 127/255), 'edgecolor': 'black'}, 
    cmap="summer", s=100
)

n = len(axes)
cols = X_dec.columns

for i in range(n):
    for j in range(n):
        ax = axes[i, j]

        if i < j:
            x = X_dec.iloc[:, j].to_numpy()
            y = X_dec.iloc[:, i].to_numpy()
            sns.kdeplot(x=x, y=y, ax=ax, levels=10, linewidths=1, color='black')

        if i < n - 1:
            ax.set_xticklabels([])
        if j > 0:
            ax.set_yticklabels([])


for j in range(n):
    axes[n-1, j].set_xlabel(
        cols[j],
        fontsize=20,
        fontname='Times New Roman',
        rotation=0,
        ha="right" 
    )

for i in range(n):
    axes[i, 0].set_ylabel(
        cols[i],
        fontsize=20,
        fontname='Times New Roman'
    )

plt.subplots_adjust(wspace=0, hspace=0)
plt.show()


In [ ]:
X_train, X_test, y_train, y_test= train_test_split(X_dec, y1, test_size=0.2, random_state=42)

In [ ]:
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

optimizers      = [Adam, RMSprop]
learning_rates  = [1e-5, 1e-4, 1e-3]
epochs_list     = [10000]
batch_sizes     = [8, 16, 24, 32, 64]
activations     = ['elu', 'relu', 'leaky_relu', 'swish', 'gelu', 'tanh']

excel_path = r"result_path"
sheet_name = "Results"

os.makedirs(os.path.dirname(excel_path), exist_ok=True)

input_dim = X_train.shape[1]

def build_model(input_dim: int, activation_name: str):
    model = Sequential()

    if activation_name == 'leaky_relu':
        model.add(Dense(256, input_shape=(input_dim,)))
        model.add(LeakyReLU())
    else:
        model.add(Dense(256, activation=activation_name, input_shape=(input_dim,)))
        
    for units in [128, 64, 32, 16, 8, 4, 2]:
        if activation_name == 'leaky_relu':
            model.add(Dense(units))
            model.add(LeakyReLU())
        else:
            model.add(Dense(units, activation=activation_name))
            
    model.add(Dense(1))
    return model

rows = []

for opt_cls in optimizers:
    for lr in learning_rates:
        for n_epochs in epochs_list:
            for bs in batch_sizes:
                for act in activations:
                    start = time.time()
                    tf.random.set_seed(SEED)

                    model = build_model(input_dim, act)

                    early_stopping = EarlyStopping(
                        monitor='val_loss',
                        patience=900,
                        mode='min',
                        restore_best_weights=True
                    )

                    model.compile(
                        optimizer=opt_cls(learning_rate=lr),
                        loss='mae',
                        metrics=['mae']
                    )

                    history = model.fit(
                        X_train, y_train,
                        epochs=n_epochs,
                        validation_split=0.2,
                        batch_size=bs,
                        callbacks=[early_stopping],
                        verbose=0
                    )

                    y_pred = model.predict(X_test, verbose=0).flatten()
                    mae = mean_absolute_error(y_test, y_pred)
                    r2  = r2_score(y_test, y_pred)

                    elapsed = time.time() - start
                    best_val_mae  = float(np.min(history.history.get('val_mae', [np.nan])))
                    best_val_loss = float(np.min(history.history.get('val_loss', [np.nan])))

                    print(
                        f"Act: {act:10s} | Opt: {opt_cls.__name__} | LR: {lr} | "
                        f"Epochs: {n_epochs} | Batch: {bs} | "
                        f"Test MAE: {mae:.4f} | R²: {r2:.4f} | "
                        f"Val MAE*: {best_val_mae:.4f} | Time: {elapsed:.2f}s"
                    )

                    rows.append({
                        "activation": act,
                        "optimizer": opt_cls.__name__,
                        "learning_rate": lr,
                        "epochs": n_epochs,
                        "batch_size": bs,
                        "test_mae": mae,
                        "test_r2": r2,
                        "best_val_mae": best_val_mae,
                        "best_val_loss": best_val_loss,
                        "fit_time_sec": elapsed
                    })

df = pd.DataFrame(rows)
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    df.to_excel(writer, index=False, sheet_name=sheet_name)

print(f"Saved report to {excel_path} with {len(df)} rows.")


In [ ]:
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['PYTHONHASHSEED'] = '0'
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

SEED = 123
start = time.time()
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

model1= Sequential([
    Dense(256, activation='swish', input_shape=(15,)),
    Dense(128, activation='swish'), 
    Dense(64, activation='swish'), 
    Dense(32, activation='swish'), 
    Dense(16, activation='swish'), 
    Dense(8, activation='swish'), 
    Dense(4, activation='swish'), 
    Dense(2, activation='linear'), 
    Dense(1)
    ])
early_stopping = EarlyStopping(monitor='val_loss', patience=900, mode='min', restore_best_weights=True)

model1.compile(optimizer= Adam(learning_rate= 0.001), loss= 'mae', metrics= ['mae'])
history= model1.fit(X_train, y_train, epochs=10000, validation_split=0.2, batch_size=16, callbacks=[early_stopping])

y_pred = model1.predict(X_test).flatten()
y_pred = np.clip(y_pred, None, 100)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print(f"Test MAE: {mae:.3f}")
print(f"Test R2: {r2:.4f}")

In [ ]:
plt.rcParams.update({
    "font.size": 40,
    "axes.labelsize": 40,
    "axes.titlesize": 40,
    "xtick.labelsize": 40,
    "ytick.labelsize": 40,
    "legend.fontsize": 40
})

rgb_color  = (38/255, 102/255, 127/255)
rgb_color2 = (239/255, 119/255, 34/255)

y_pred = pd.DataFrame(model1.predict(X_test))
y_true = pd.DataFrame(y_test)

y_true_flat = np.ravel(np.array(y_true))
y_pred_flat = np.ravel(np.array(y_pred))
abs_errors  = np.abs(y_true_flat - y_pred_flat)
errors      = y_pred_flat - y_true_flat

fig, axs = plt.subplots(2, 2, figsize=(20, 20), constrained_layout=True)

axs[0, 0].plot(history.history['loss'], label='Training Loss', lw=3, color=rgb_color)
axs[0, 0].plot(history.history['val_loss'], label='Validation Loss', lw=3, color=rgb_color2)
axs[0, 0].set_xlabel('Epochs')
axs[0, 0].set_ylabel('Mean Absolute Error')
axs[0, 0].set_title("", pad=15)
axs[0, 0].grid(True)
axs[0, 0].legend()

axs[1, 0].hist(abs_errors, bins=15, color=rgb_color, edgecolor='black')
axs[1, 0].set_xlabel("Absolute Error")
axs[1, 0].set_ylabel("Frequency")
axs[1, 0].set_title("", pad=15)
axs[1, 0].grid(True)

axs[0, 1].scatter(
    y_true_flat,
    y_pred_flat,
    s=200,
    color=rgb_color,
    alpha=1
)

lims = [
    min(np.min(y_true_flat), np.min(y_pred_flat)),
    max(np.max(y_true_flat), np.max(y_pred_flat))
]
axs[0, 1].plot(lims, lims, 'k--', lw=2.5)
axs[0, 1].set_xlim(lims)
axs[0, 1].set_ylim(lims)
axs[0, 1].set_xlabel('True (y_test)')
axs[0, 1].set_ylabel('Predicted (y_pred)')
axs[0, 1].set_title("", pad=10)
axs[0, 1].grid(True)

axs[1, 1].scatter(
    y_pred_flat,
    errors,
    s=200,
    color=rgb_color2,
    alpha=1
)
axs[1, 1].axhline(0, color='black', ls='--', lw=2.5)
axs[1, 1].set_xlabel('Predicted (y_pred)')
axs[1, 1].set_ylabel('Error (y_pred - y_true)')
axs[1, 1].set_title("", pad=15)
axs[1, 1].grid(True)

plt.show()


In [ ]:
def pred(x):
    x= pd.DataFrame(sc.transform(x), columns=X.columns)
    x1=x.drop(['Epoxy Descriptor 4', 'Catalyst Descriptor 2', 
                         'Catalyst Descriptor 4', 'Catalyst (%wt)', 'Alternate Tg (C)', 'Alternate Temperature', 'Temperature (C)',
                         'Required Time (min)', 'Additive Descriptor 5'], axis=1)
    x_q= pd.DataFrame(pt.transform(x1))
    x_dec= pd.DataFrame(pca1.transform(x_q))
    return(model1.predict(x_dec))

In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 30

times = np.arange(1, 60, 2)
effs1, effs2 = [], []

for T in times:
    X_time = [[6.9429717, 4.38475168, 0.561047652, -4.564026157, -0.706662196, 
               1.997871842, 5.854876201, -5.803161689, -0.504273323, 1.674221774,
               1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
               -71, 71, 0.0077, -129, 90, math.sqrt(T), T]]
    effs1.append(float(pred(X_time)))

    X_time_1 = [[7.445151299, 3.393759273, -0.926463773, -4.801720504, -1.054142876,  
                 0.741334851, -0.676684554, 2.519530414, 1.356952602, -0.010481245, 1,
                 -0.134635561, -0.175244042, 1.773563805, -2.633995277, -1.184607486,
                 -8, 8, -7.250192946, 3.072477777, -2.096761108, 4.302690114,
                 -4.313043074, 1,
                 -45, 45, 0.0256, -39, 180, math.sqrt(T), T]]
    effs2.append(float(pred(X_time_1)))

color1 = (0/255, 0/255, 0/255)
color2 = (249/255, 89/255, 18/255)

fig, ax1 = plt.subplots(figsize=(10, 10))

ax1.scatter(times, effs1, color=color1, s=140)
ax1.plot(times, effs1, color=color1, linewidth=2, label="DGEBA + 2-AFD at 90℃")
ax1.set_xlabel("Time (min)", fontsize=30)
ax1.set_ylabel("Self-healing Efficiency (%)", fontsize=30, color=color1)
ax1.tick_params(axis='both', labelsize=21)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim(85, 97.5)

ax2 = ax1.twinx()
ax2.scatter(times, effs2, color=color2, s=140)
ax2.plot(times, effs2, color=color2, linewidth=2, label="DGEBA + Adipic Acid + TBD & CNTs at 90℃")
ax2.set_ylabel("Self-healing Efficiency (%)", fontsize=30, color=color2)
ax2.tick_params(axis='y', labelsize=30, labelcolor=color2)
ax2.set_ylim(45, 57.5)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(lines1 + lines2, labels1 + labels2,
           loc="center left", bbox_to_anchor=(0.01, 0.95),
           fontsize=18.5, frameon=True)

ax1.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 26

times = np.arange(70, 100, 2)
effs1, effs2 = [], []

for T in times:
    X_time = [[6.9429717, 4.38475168, 0.561047652, -4.564026157, -0.706662196, 
               1.997871842, 5.854876201, -5.803161689, -0.504273323, 1.674221774,
               1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
               -71, 71, (-1/(T-219)), T-219, T, 7.746, 60]]
    effs1.append(float(pred(X_time)))

    X_time_1 = [[7.445151299, 3.393759273, -0.926463773, -4.801720504, -1.054142876,  
                 0.741334851, -0.676684554, 2.519530414, 1.356952602, -0.010481245, 1,
                 -0.134635561, -0.175244042, 1.773563805, -2.633995277, -1.184607486,
                 -8, 8, -7.250192946, 3.072477777, -2.096761108, 4.302690114,
                 -4.313043074, 1,
                 -45, 45, (-1/(T-219)), T-219, T, 7.746, 60]]
    effs2.append(float(pred(X_time_1)))


color1 = (0/255, 0/255, 0/255)
color2 = (249/255, 89/255, 18/255)

fig, ax1 = plt.subplots(figsize=(10.5, 10))

ax1.scatter(times, effs1, color=color1, s=140)
ax1.plot(times, effs1, color=color1, linewidth=2, label="DGEBA + 2-AFD for 1 hour")
ax1.set_xlabel("Temperature (℃)", fontsize=30)
ax1.set_ylabel("Self-healing Efficiency (%)", fontsize=30, color=color1)
ax1.tick_params(axis='both', labelsize=30)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim(96.5, 98)

ax2 = ax1.twinx()
ax2.scatter(times, effs2, color=color2, s=140)
ax2.plot(times, effs2, color=color2, linewidth=2, label="DGEBA + Adipic Acid + TBD & CNTs for 1 hour")
ax2.set_ylabel("Self-healing Efficiency (%)", fontsize=30, color=color2)
ax2.tick_params(axis='y', labelsize=30, labelcolor=color2)
ax2.set_ylim(47.5, 48.5)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="center left",
    bbox_to_anchor=(0.01, 0.95),
    fontsize=18.5,
    frameon=True
)

ax1.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()
